# 43 机器学习补充方法

> **模块**：机器学习 ｜ **数据**：无（内置示例） ｜ **依赖**：matplotlib、seaborn、numpy、scikit-learn、pandas

## 1. 问题背景

高斯混合模型做概率密度聚类，高斯过程回归给出带不确定性的预测，价值迭代求解马尔可夫决策过程。三者分别代表概率建模、贝叶斯回归与序贯决策的常用工具。

## 2. 学习目标

- 理解 GMM 的 EM 迭代与协方差结构
- 会用高斯过程回归给出预测区间
- 掌握价值迭代的收敛条件

## 3. 环境与数据

- 依赖：matplotlib、seaborn、numpy、scikit-learn、pandas
- 数据：无（内置示例）

## 4. 方法与实现

### 4.1 高斯混合模型（GMM）

代码来自sklearn文档

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()
import numpy as np

#产生实验数据
from sklearn.datasets.samples_generator import make_blobs
X, y_true = make_blobs(n_samples=400, centers=4,
                       cluster_std=0.60, random_state=0)
X = X[:, ::-1] #交换列是为了方便画图

from sklearn.mixture import GaussianMixture
gmm = GaussianMixture(n_components=4).fit(X)
labels = gmm.predict(X)
plt.scatter(X[:, 0], X[:, 1], c=labels, s=40, cmap='viridis')

In [ ]:
from matplotlib.patches import Ellipse

def draw_ellipse(position, covariance, ax=None, **kwargs):
    """用给定的位置和协方差画一个椭圆"""
    ax = ax or plt.gca()

    #将协方差转换为主轴
    if covariance.shape == (2, 2):
        U, s, Vt = np.linalg.svd(covariance)
        angle = np.degrees(np.arctan2(U[1, 0], U[0, 0]))
        width, height = 2 * np.sqrt(s)
    else:
        angle = 0
        width, height = 2 * np.sqrt(covariance)

    #画出椭圆
    for nsig in range(1, 5):
        ax.add_patch(Ellipse(position, nsig * width, nsig * height,
                             angle, **kwargs))

def plot_gmm(gmm, X, label=True, ax=None):
    ax = ax or plt.gca()
    
    labels = gmm.fit(X).predict(X)
    if label:
        ax.scatter(X[:, 0], X[:, 1], c=labels, s=40, cmap='viridis', zorder=2)
    else:
        ax.scatter(X[:, 0], X[:, 1], s=40, zorder=2)
    ax.axis('equal')
    
    w_factor = 0.2 / gmm.weights_.max()
    for pos, covar, w in zip(gmm.means_, gmm.covariances_, gmm.weights_):
        draw_ellipse(pos, covar, alpha=w * w_factor)


#用椭圆形来拟合数据
rng = np.random.RandomState(13)
X_stretched = np.dot(X, rng.randn(2, 2))
gmm = GaussianMixture(n_components=4, covariance_type='full', random_state=42)
plot_gmm(gmm, X_stretched)

### 4.2 高斯过程回归

代码来自[参考网址](https://blog.csdn.net/qq_40597317/article/details/80889114)

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

np.random.seed(1)


def f(x):
    """The function to predict."""
    return x * np.sin(x)

# ----------------------------------------------------------------------
#  First the noiseless case
X = np.atleast_2d([1., 3., 5., 6., 7., 8.]).T

# Observations
# similar to flatten
y = f(X).ravel() 

# Mesh the input space for evaluations of the real function, the prediction and
# its MSE
x = np.atleast_2d(np.linspace(0, 10, 1000)).T

# Instantiate a Gaussian Process model
kernel = C(1.0, (1e-3, 1e3)) * RBF(10, (1e-2, 1e2))
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=9)

# Fit to data using Maximum Likelihood Estimation of the parameters
gp.fit(X, y)

# Make the prediction on the meshed x-axis (ask for MSE as well)
y_pred, sigma = gp.predict(x, return_std=True)

# Plot the function, the prediction and the 95% confidence interval based on
# the MSE
plt.figure()
plt.plot(x, f(x), 'r:', label=u'$f(x) = x\ sin(x)$')
plt.plot(X, y, 'r.', markersize=10, label=u'Observations')
plt.plot(x, y_pred, 'b-', label=u'Prediction')
plt.fill(np.concatenate([x, x[::-1]]),
         np.concatenate([y_pred - 1.9600 * sigma,
                        (y_pred + 1.9600 * sigma)[::-1]]),
         alpha=.5, fc='b', ec='None', label='95% confidence interval')
plt.xlabel('$x$')
plt.ylabel('$f(x)$')
plt.ylim(-10, 20)
plt.legend(loc='upper left')

# ----------------------------------------------------------------------
# now the noisy case
X = np.linspace(0.1, 9.9, 20)
X = np.atleast_2d(X).T

# Observations and noise
y = f(X).ravel()
dy = 0.5 + 1.0 * np.random.random(y.shape)
noise = np.random.normal(0, dy)
y += noise

# Instantiate a Gaussian Process model
gp = GaussianProcessRegressor(kernel=kernel, alpha=dy ** 2,
                              n_restarts_optimizer=10)

# Fit to data using Maximum Likelihood Estimation of the parameters
gp.fit(X, y)

# Make the prediction on the meshed x-axis (ask for MSE as well)
y_pred, sigma = gp.predict(x, return_std=True)

# Plot the function, the prediction and the 95% confidence interval based on
# the MSE
plt.figure()
plt.plot(x, f(x), 'r:', label=u'$f(x) = x\,\sin(x)$')
plt.errorbar(X.ravel(), y, dy, fmt='r.', markersize=10, label=u'Observations')
plt.plot(x, y_pred, 'b-', label=u'Prediction')
plt.fill(np.concatenate([x, x[::-1]]),
         np.concatenate([y_pred - 1.9600 * sigma,
                        (y_pred + 1.9600 * sigma)[::-1]]),
         alpha=.5, fc='b', ec='None', label='95% confidence interval')
plt.xlabel('$x$')
plt.ylabel('$f(x)$')
plt.ylim(-10, 20)
plt.legend(loc='upper left')

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
from sklearn.gaussian_process import GaussianProcessRegressor
a=np.random.random(50).reshape(50,1)
b=a*2+np.random.random(50).reshape(50,1)
plt.scatter(a,b,marker = 'o', color = 'r', label='3', s = 15)
plt.show()
gaussian=GaussianProcessRegressor()
fiting=gaussian.fit(a,b)
c=np.linspace(0.1,1,100)
d=gaussian.predict(c.reshape(100,1))
plt.scatter(a,b,marker = 'o', color = 'r', label='3', s = 15)
plt.plot(c,d)
plt.show()

### 4.3 马尔可夫决策过程（价值迭代）

In [ ]:
from IPython.display import Latex
from IPython.display import display, Math, Latex
print_latex = lambda latex_str: display(Latex(latex_str))

print_latex(r'$\left\{\begin{array}{l}R(s)=\left\{\begin{array}{l}+1,\ s=(4,3)\\-1,\ s=(4,2)\\-0.02\ ,\ otherwise.\end{array}\right.\\\
    \gamma=0.99\\\
    正常走概率0.8, 向左向右0.1\
    \end{array}\right.$')

In [ ]:
# 价值迭代算法

import pandas as pd
import numpy as np
import pprint

max_x = 3
max_y = 2
min_x = 0
min_y = 0
direc = ['u', 'r', 'd', 'l']
a_size = len(direc)
offset = {'u': (0, 1), 'd': (0, -1), 'l': (-1, 0), 'r': (1, 0)}
p_normal = 0.8
p_abnormal = 0.1
R_rest = -0.02
gamma = 0.99
e = 1e-30
obstacle = {(1, 1)}
states = {(x, y) for x in range(min_x, max_x+1) for y in range(min_y, max_y+1)} - obstacle
next_state = {}
for s in states:
    if s[0] != 3 or s[1] != 2:
        for a in direc:
            if a == 'u' and s[1] == max_y or a == 'd' and s[1] == min_y or a == 'l' and s[0] == min_x or a == 'r' and s[0] == max_x:
                next_state[s + (a,)] = s
            else: 
                if tuple(np.add(s, offset[a])) in obstacle:
                    next_state[s + (a,)] = s
                else:
                    next_state[s + (a,)] = tuple(np.add(s, offset[a]))
    else:
        for a in direc:
            next_state[s + (a,)] = s

            
# Reward Function
R = np.ones((max_x-min_x+1, max_y-min_y+1)) * R_rest
R[3][2] = 1
R[3][1] = -1
Rs = {s: R[s[0]][s[1]] for s in states}
Rs.update({s: 0 for s in obstacle})

# Value Function
V = np.zeros((max_x-min_x+1, max_y-min_y+1))
Vs = {s: V[s[0]][s[1]] for s in states}
Vs.update({s: -np.inf for s in obstacle})

# Get Value Function
n = 0
while True:
    lastV = pd.Series(Vs)
    for s in states:
        max_next = -np.inf
        if s == (3, 2):
            max_next = 0
        else:
            for i in range(len(direc)):
                cur_next = p_normal * Vs[next_state[s + (direc[i],)]] \
                            + p_abnormal*Vs[next_state[s + (direc[(i+1)%a_size],)]] \
                            + p_abnormal*Vs[next_state[s + (direc[(i+a_size-1)%a_size],)]]
                if cur_next > max_next:
                    max_next = cur_next
        Vs[s] = Rs[s] + gamma*max_next
    V = pd.Series(Vs)
    n += 1
    if np.sum(np.square(lastV-V)) < e:
        break
    
# Get Policy
policy = {}
for s in states:
        max_next = -np.inf
        best_policy = direc[0]
        if s == (3, 2):
            policy[s] = 'Stop'
        else:
            for i in range(len(direc)):
                cur_next = p_normal * Vs[next_state[s + (direc[i],)]] \
                            + p_abnormal*Vs[next_state[s + (direc[(i+1)%a_size],)]] \
                            + p_abnormal*Vs[next_state[s + (direc[(i+a_size-1)%a_size],)]]
                if cur_next > max_next:
                    best_policy = direc[i]
                    max_next = cur_next
            policy[s] = best_policy

print(n)
V = np.array([[Vs[(x, y)] for x in range(min_x, max_x+1)] for y in range(min_y, max_y+1)])[::-1]
print(V)
Po = np.array([[policy[(x, y)] if (x, y) not in obstacle else None for x in range(min_x, max_x+1)] for y in range(min_y, max_y+1)])[::-1]
print(Po)

In [ ]:
# 价值迭代算法

import pandas as pd
import numpy as np
import pprint

max_x = 3
max_y = 2
min_x = 0
min_y = 0
direc = ['u', 'r', 'd', 'l']
a_size = len(direc)
offset = {'u': (0, 1), 'd': (0, -1), 'l': (-1, 0), 'r': (1, 0)}
p_normal = 0.8
p_abnormal = 0.1
R_rest = -0.02
gamma = 0.99
e = 1e-30
obstacle = {(1, 1)}
states = {(x, y) for x in range(min_x, max_x+1) for y in range(min_y, max_y+1)} - obstacle
next_state = {}
for s in states:
    if s[0] != 3 or s[1] != 2:
        for a in direc:
            if a == 'u' and s[1] == max_y or a == 'd' and s[1] == min_y or a == 'l' and s[0] == min_x or a == 'r' and s[0] == max_x:
                next_state[s + (a,)] = s
            else: 
                if tuple(np.add(s, offset[a])) in obstacle:
                    next_state[s + (a,)] = s
                else:
                    next_state[s + (a,)] = tuple(np.add(s, offset[a]))
    else:
        for a in direc:
            next_state[s + (a,)] = s

            
# Reward Function
R = np.ones((max_x-min_x+1, max_y-min_y+1)) * R_rest
R[3][2] = 1
R[3][1] = -1
Rs = {s: R[s[0]][s[1]] for s in states}
Rs.update({s: 0 for s in obstacle})

# Value Function
V = np.zeros((max_x-min_x+1, max_y-min_y+1))
Vs = {s: V[s[0]][s[1]] for s in states}
Vs.update({s: -np.inf for s in obstacle})

# Get Value Function
n = 0
while True:
    lastV = pd.Series(Vs)
    for s in states:
        max_next = -np.inf
        if s == (3, 2):
            max_next = 0
        else:
            for i in range(len(direc)):
                cur_next = p_normal * Vs[next_state[s + (direc[i],)]] \
                            + p_abnormal*Vs[next_state[s + (direc[(i+1)%a_size],)]] \
                            + p_abnormal*Vs[next_state[s + (direc[(i+a_size-1)%a_size],)]]
                if cur_next > max_next:
                    max_next = cur_next
        Vs[s] = Rs[s] + gamma*max_next
    V = pd.Series(Vs)
    n += 1
    if np.sum(np.square(lastV-V)) < e:
        break
    
# Get Policy
policy = {}
for s in states:
        max_next = -np.inf
        best_policy = direc[0]
        if s == (3, 2):
            policy[s] = 'Stop'
        else:
            for i in range(len(direc)):
                cur_next = p_normal * Vs[next_state[s + (direc[i],)]] \
                            + p_abnormal*Vs[next_state[s + (direc[(i+1)%a_size],)]] \
                            + p_abnormal*Vs[next_state[s + (direc[(i+a_size-1)%a_size],)]]
                if cur_next > max_next:
                    best_policy = direc[i]
                    max_next = cur_next
            policy[s] = best_policy

print(n)
V = np.array([[Vs[(x, y)] for x in range(min_x, max_x+1)] for y in range(min_y, max_y+1)])[::-1]
print(V)
Po = np.array([[policy[(x, y)] if (x, y) not in obstacle else None for x in range(min_x, max_x+1)] for y in range(min_y, max_y+1)])[::-1]
print(Po)

## 5. 小结

三种方法都显式建模不确定性；超参数、核函数与收敛阈值直接影响结果，应结合验证集判断。

> 内容来源：教材参考实现《数学建模算法与应用》练习稿（MIT，Zhanghan Wang），原始文件已并入本篇，见 `THIRD-PARTY.md`。